### Association Rules using profiles

In [6]:
import pandas as pd
import pickle
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

In [2]:
profiles = pd.read_parquet('data/profiles.parquet')

In [8]:
# convert to list of lists
transactions = profiles['favorites_anime'].tolist()

# encode into boolean matrix
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)

print(df_encoded.shape)  # (users, unique_anime)

(32398, 4647)


In [10]:
# min_support = 0.01 means the itemset must appear in at least 1% of users
frequent_itemsets = fpgrowth(df_encoded, min_support=0.01, use_colnames=True)

rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.2)
rules = rules.sort_values('lift', ascending=False)

print(f"Rules found: {len(rules)}")
rules.head()

Rules found: 64


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
1,(1575),(2904),0.083308,0.055991,0.014445,0.173398,3.096877,1.0,0.009781,1.142035,0.738627,0.115698,0.124370,0.215695
0,(2904),(1575),0.055991,0.083308,0.014445,0.257993,3.096877,1.0,0.009781,1.235423,0.717254,0.115698,0.190561,0.215695
36,(1),(30),0.071980,0.068584,0.013396,0.186106,2.713534,1.0,0.008459,1.144395,0.680456,0.105340,0.126176,0.190713
37,(30),(1),0.068584,0.071980,0.013396,0.195320,2.713534,1.0,0.008459,1.153278,0.677976,0.105340,0.132906,0.190713
17,(5114),(11061),0.146460,0.094142,0.030403,0.207587,2.205050,1.0,0.016615,1.143164,0.640269,0.144640,0.125235,0.265269


In [15]:
animes = pd.read_parquet('data/animes.parquet')

In [104]:
def give_associations(title, animes, rules):
    uid = animes[animes.title == title].iloc[0]['uid']
    selected = [str(uid)]
    
    recs = rules[rules['antecedents'] == frozenset(selected)]
    
    associations = []
    for r in recs.consequents:
        id = list(r)[0]
        match = animes[animes.uid == int(id)]  # cast to int
        if not match.empty:
            associations.append(match.iloc[0])
    
    return pd.DataFrame(associations)[['title', 'uid','img_url']]  

    

In [115]:
give_associations('Hunter x Hunter (2011)',animes,rules)

,title,uid,img_url
3,Fullmetal Alchemist: Brotherhood,5114,https://cdn.myanimelist.net/images/anime/1223/...
707,One Piece,21,https://cdn.myanimelist.net/images/anime/6/732...
746,Tengen Toppa Gurren Lagann,2001,https://cdn.myanimelist.net/images/anime/4/512...
773,Steins;Gate,9253,https://cdn.myanimelist.net/images/anime/5/731...
740,Death Note,1535,https://cdn.myanimelist.net/images/anime/9/945...


In [107]:
rules.to_pickle('models/rules.pkl')